# Capstone build --- Chapter 6: Safe Tool Execution

In Chapter~1 the environment ran a tool as soon as the arguments validated against its schema. A regulated agent cannot execute every well-formed call: a message carrying a Social Security number must not be forwarded to a model, and a prompt-injection attempt must not reach a tool at all. Chapter~6 inserts a gate stack between the proposal and the tool body. The `GovernedToolExecutor` runs each gate in turn and refuses the call if any gate denies it, so an unsafe action is stopped before it acts rather than after.

## The action space, as before

The registry is the five-tool action space from Chapter~5. Here it is built with the classify tool alone, which is enough to show the gate stack refusing a call; the full registry behaves identically because the gates act on the proposed action, not on which tool it names.

In [ ]:
from forgeloop.agents.tools import ToolRegistry
from forgeloop.agents.capstone.banking_tools import classify_complaint

registry = ToolRegistry()
registry.register(classify_complaint)

## The gate stack

The executor is configured with an ordered list of gates. The `SyntaxGate` rejects a call that does not match the tool's schema. The policy gate is built from a `PolicyEngine`, which aggregates the banking policies; the deterministic PII policy is used here, a regex format check that is the right tool for detecting an SSN, card number or email. The shipped harness adds two Qwen-backed semantic policies (prompt injection and prohibited advice) and the trained GMS plausibility gate of Chapter~12 to this same list.

In [ ]:
from forgeloop.agents.tools import GovernedToolExecutor
from forgeloop.agents.governance import PolicyEngine, SyntaxGate, pii_policy

engine = PolicyEngine([pii_policy])
gates = [SyntaxGate(), engine.as_gate()]
executor = GovernedToolExecutor(registry, gates=gates)
print('gate stack:', [g.__class__.__name__ for g in gates])

## A safe call runs; an unsafe call is denied

The executor returns a `ToolResult` either way. A well-formed, policy-clean call runs the tool and reports `success=True`. A call whose message carries an SSN is denied by the policy gate: the result is `success=False`, its error names the gate that refused it, and the tool body never ran.

In [ ]:
from forgeloop.agents.core.action import ToolCall

clean = executor.execute(ToolCall(
    tool_name='classify_complaint',
    arguments={'message': 'I was charged a $35 overdraft fee I did not authorize.'},
))
print('clean call : success=', clean.success, '| output=', clean.output)

unsafe = executor.execute(ToolCall(
    tool_name='classify_complaint',
    arguments={'message': 'My SSN is 123-45-6789 and I want a refund.'},
))
print('unsafe call: success=', unsafe.success, '| error=', unsafe.error)
for r in unsafe.gate_results:
    print('   gate', r.gate_name, '->', r.decision)

This governed executor replaces the bare environment of Chapter~1. Every tool call the agent proposes now passes the gate stack first, and a denied call returns a failed result the agent can act on --- in the capstone, a failed result at any step drives an escalation (Chapter~14). Chapter~7 bounds the loop in cost and latency, and Chapter~12 completes the gate stack with the trained GMS plausibility gate and wraps the executor in the governance harness.